# `ammico` workshop November 2026

*Delia Dumitrescu, Inga Ulusoy, Dmitrii Kapitan*

With `ammico`, you can analyze text on images and image content at the same time, see https://ssciwr.github.io/AMMICO/. First we start by importing the packages.

In [ ]:
import os
from pprint import pprint

import matplotlib.pyplot as plt
from PIL import Image

import ammico

This imports all the functionality from `ammico`. To analyze images, you can manually upload images to the Jupyter server; in this workshop, we will use pre-prepared datasets in `/home/jovyan/data_mounts/webdav`.

# Step 1: Read your data into AMMICO

`ammico` reads in files from a directory. You can iterate through directories in a recursive manner and filter by extensions. Note that the order of the files may vary on different OS. Reading in these files creates a dictionary `image_dict`, with one entry per image file, containing the file path and filename. This dictionary is the main data structure that ammico operates on and is extended successively with each detector run as explained below.

For reading in the files, the ammico function `find_files` is used, with optional keywords:

| input key | input type | possible input values |
| --------- | ---------- | --------------------- |
`path` | `str` | the directory containing the image files (defaults to the location set by environment variable `AMMICO_DATA_HOME`) |
| `pattern` | `str\|list` | the file extensions to consider (defaults to "png", "jpg", "jpeg", "gif", "webp", "avif", "tiff") |
| `recursive` | `bool` | include subdirectories recursively (defaults to `True`) |
| `limit` | `int` | maximum number of files to read (defaults to `20`, for all images set to `None` or `-1`) |
| `random_seed` | `int` | the random seed for shuffling the images; applies when only a few images are read and the selection should be preserved (defaults to `None`) |

In [ ]:
# Define your data path
data_path = (
    "/home/jovyan/data_mounts/webdav/dataset_politicians/"  # the selected directory
)

# Find files and create the image dictionary
image_dict = ammico.find_files(
    path=data_path,
    limit=-1,  # Limit the number of files to process (optional)
)

# 2. Run the content analysis: Image summary

As an example we will create an image caption ("Summary") using a vision-language model from the [QWEN 2.5 Vision-Language model family](https://huggingface.co/collections/Qwen/qwen25-vl).

`ammico` uses a model as hosted externally and reached over an **OpenAI-compatible HTTP API**. The same code works against:

* a self-hosted server such as [vLLM](https://docs.vllm.ai/) serving `Qwen/Qwen2.5-VL-7B-Instruct`,
* the OpenAI API (`https://api.openai.com/v1`),
* Google Gemini via its OpenAI-compatibility endpoint (`https://generativelanguage.googleapis.com/v1beta/openai/`).

Only three settings change between them: the base URL, the API key and the model id. They are read from the environment variables `AMMICO_API_BASE_URL`, `AMMICO_API_KEY` and `AMMICO_MODEL_ID` (you can also pass them directly to `InferenceModel(...)`). The `openai` client is an optional dependency — install it with `pip install ammico[api]`.

In [ ]:
# Configure the externally hosted vision-language model (OpenAI-compatible endpoint).
# For a self-hosted vLLM server, AMMICO_API_KEY is the value passed to `vllm serve --api-key`.
os.environ["AMMICO_API_BASE_URL"] = (
    # "http://localhost:8000/v1"  # using vllm
    # "http://localhost:11434/v1"  # using ollama
    "http://ammico.ammico.svc:80/v1"  # using Jupyter4NFDI
)
os.environ["AMMICO_API_KEY"] = "EMPTY"  # using Jupyter4NFDI
os.environ["AMMICO_MODEL_ID"] = "Qwen/Qwen2.5-VL-3B-Instruct"  # using Jupyter4NFDI
os.environ["AMMICO_PRIVACY_ACK"] = "True"

model = ammico.InferenceModel()  # reads the AMMICO_API_* environment variables

Then, we create an instance of the Python class that handles the image summary and visual question answering tasks:

In [ ]:
image_summary_vqa = ammico.ImageSummaryDetector(summary_model=model, subdict=image_dict)

After this, we can create the image captions. Depending on the number of images and the hardware provided, this can take several minutes.

In [ ]:
summary = image_summary_vqa.analyse_images_from_dict(
    analysis_type="summary", is_concise_summary=True
)

The results are provided in the updated dictionary. Execute the cell below to see the image that was analyzed together with the generated caption (summary).

In [ ]:
for key in summary:
    # Load and display the image
    image_path = summary[key]["filename"]
    img = Image.open(image_path)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")  # Hide axes
    plt.title(f"Summary: {summary[key]['caption']}", fontsize=12)
    plt.tight_layout()
    plt.show()

# 3. Run the content analysis: Visual question answering

You may also ask questions about the images. For this, provide a list of questions and pass it to the Python class that you have instantiated above:

In [ ]:
list_of_questions = [
    "Is there a politician in the picture?",
    "Is there a politician in the picture? Answer yes, no or dk. Don't guess.",
    "Is there a politician in the picture? Answer yes, no, or dk. Briefly justify your answer and, if there is a politician, give the name. Don't guess.",
    "Is any of the following politicians in the picture: Donald Trump, Volodymyr Zelenskyy or Vladimir Putin? Answer yes or no. If yes, give the name. Don't guess.",
]  # add or replace with your own questions

In [ ]:
answers = image_summary_vqa.analyse_images_from_dict(
    analysis_type="questions",
    list_of_questions=list_of_questions,
    is_concise_summary=True,
    is_concise_answer=True,
)

Again for your convenience we display the images and the answers to the questions together below:

In [ ]:
for key in answers:
    # Load and display the image
    image_path = answers[key]["filename"]
    img = Image.open(image_path)
    for answer in answers[key]["vqa"]:
        pprint(answer, width=100, compact=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")  # Hide axes
    plt.tight_layout()
    plt.show()

# 4. Export the results

To export the results for further processing, convert the image dictionary into a pandas dataframe.

In [ ]:
image_df = ammico.get_dataframe(image_dict)

Inspect the dataframe:

In [ ]:
image_df.head(5)

Export the dataframe to a csv file:

In [ ]:
image_df.to_csv("./politicians_data_out.csv")

# 5. Analyse the results
You can compare the results to the manual labels and observe how accurate the VLM captures the content.

## Question accuracy for one model

Which of several questions (prompts) gets the most images right, compared with the annotated ground truth?

Fill in the two paths in the next cell and run all cells. Tables and plots are saved in `comparison_results/` next to the (first) result file.

In [ ]:
# ============ SETTINGS: the only cell you need to change ============
RESULTS_FILE = "./politicians_data_out.csv"  # csv file with the answers of ONE model (your ammico output)
IMAGE_DIR = data_path  # folder with the images, or the above path in part 1

**Datasets** – the dataset (emotions, politicians or symbols) is detected from the image names in the result file, so only the paths change. 

The images can be in one flat folder; they are only needed for the image examples. Questions, ground truth and expected answers are defined in `DATASETS` in `vqa_eval_utils.py`; the ground-truth and question files are read from `ground_truth/` next to the image folder (or from the `*_output/` folders of the course repository). The answers must be to the questions in these files, in the same order. A file with several models (e.g. the coded symbols table) only works in `question_accuracy_models.ipynb`.

## 5.1 Data and questions

In [ ]:
import vqa_eval_utils as U

run = U.load(RESULTS_FILE, IMAGE_DIR, "question_accuracy_single_model")
assert len(run.models) == 1, (
    f"the file contains several models {run.models}: use question_accuracy_models.ipynb"
)
U.show_questions(run)

## 5.2 From free text to answers
Each answer is mapped to a category (yes / no / dk, …) by the rules of its question type. `exact`, `first` and `package_code` are safe; the keyword rules (`kw_*`) are heuristics, so read the sample of raw answers below against their `normalised` value.

In [ ]:
U.show_normalisation(run)

## 5.3 Accuracy per question
**Accuracy** = share of images whose (normalised) answer matches the expected answer from the ground truth. DK, unclear and unparsed answers count as **wrong**; images without an expected answer (e.g. label `unclear`) are not scored.
The **majority baseline** is the accuracy of always giving the most frequent expected answer – a question is only useful if it beats it clearly. Error bars are 95% Wilson confidence intervals.

In [ ]:
acc = U.show_accuracy(run)

## 5.4 Accuracy per image group
The same accuracy for each image group: a low value shows which kind of image makes a question fail.

In [ ]:
U.show_groups(run, acc)

## 5.5 Hardest images
The images with the largest share of wrong answers. For each image: the image, the answer generated for every question (`read as` = the normalised answer that is scored), the expected answer, and which answers are wrong. Errors on every question point to a hard image or a questionable label; errors on some questions point to the wording.

In [ ]:
U.show_hardest_images(run, n=6)  # n = number of images shown